In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('outage.csv', skiprows=5)
df = df.drop('variables', axis=1)
df = df.iloc[1:]

In [5]:
df.isna().sum()[df.isna().sum() > 0]

,0
MONTH,9
CLIMATE.REGION,6
ANOMALY.LEVEL,9
CLIMATE.CATEGORY,9
OUTAGE.START.DATE,9
OUTAGE.START.TIME,9
OUTAGE.RESTORATION.DATE,58
OUTAGE.RESTORATION.TIME,58
CAUSE.CATEGORY.DETAIL,471
HURRICANE.NAMES,1462


CUSTOMERS.AFFECTED could be MNAR. This data could fail to be reported when the magnitude of the impact is very large or very small.

In [6]:
col = "DEMAND.LOSS.MW"
df = df.copy()

# Missingness indicator: True if DEMAND.LOSS.MW is missing
df["miss_demand"] = df[col].isna()
df["miss_demand"].value_counts(dropna=False)

,count
miss_demand,
False,829
True,705


In [7]:
df["OUTAGE.DURATION"].dtype
df["OUTAGE.DURATION"].head(10)
df["OUTAGE.DURATION"].astype(str).str.slice(0, 40).value_counts().head(10)

,count
OUTAGE.DURATION,
1,97
0,78
nan,58
2880,15
300,14
60,14
1440,13
15,9
4320,9


In [8]:
df["OUTAGE.DURATION_NUM"] = pd.to_numeric(df["OUTAGE.DURATION"], errors="coerce")
df["OUTAGE.DURATION_NUM"].describe()

,OUTAGE.DURATION_NUM
count,1476.000000
mean,2625.398374
std,5942.483307
min,0.000000
25%,102.250000
50%,701.000000
75%,2880.000000
max,108653.000000


In [9]:
x = "OUTAGE.DURATION_NUM"

tmp = df.loc[df[x].notna(), ["miss_demand", x]].copy()

obs = (
    tmp.loc[tmp["miss_demand"], x].mean()
    - tmp.loc[~tmp["miss_demand"], x].mean()
)

obs


def perm_test_diff_means(tmp, group_col, value_col, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)
    g = tmp[group_col].to_numpy()
    v = tmp[value_col].to_numpy()

    obs = v[g].mean() - v[~g].mean()

    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        g_perm = rng.permutation(g)
        perm_stats[i] = v[g_perm].mean() - v[~g_perm].mean()

    # two-sided p-value
    p = (np.abs(perm_stats) >= np.abs(obs)).mean()
    return obs, perm_stats, p

obs_stat, perm_stats, pval = perm_test_diff_means(tmp, "miss_demand", x, n_perm=5000, seed=42)
obs_stat, pval

(np.float64(128.76779051172707), np.float64(0.6824))

There is no statistical evidence that missingness of demand.loss.mw depends on outage duration. (p=0.6824 >= 0.05)

In [10]:
import plotly.express as px
import numpy as np

plot_df = tmp.copy()
plot_df["group"] = plot_df["miss_demand"].map({
    True: "DEMAND.LOSS.MW missing",
    False: "DEMAND.LOSS.MW present"
})

plot_df["log_duration"] = np.log1p(plot_df[x])

fig = px.histogram(
    plot_df,
    x="log_duration",
    color="group",
    nbins=60,
    barmode="overlay",
    histnorm="probability density",
    title="Log Outage Duration by DEMAND.LOSS.MW Missingness"
)

fig.update_layout(
    xaxis_title="log(1 + OUTAGE.DURATION)",
    yaxis_title="Density"
)

fig.show()

In [11]:
from scipy.stats import chi2_contingency

cat = "CAUSE.CATEGORY"

tmpc = df.loc[df[cat].notna(), ["miss_demand", cat]].copy()

In [12]:
ct = pd.crosstab(tmpc[cat], tmpc["miss_demand"])
chi2_obs = chi2_contingency(ct, correction=False)[0]
chi2_obs

np.float64(101.3661710576918)

In [13]:
import numpy as np

def perm_test_chi2(tmpc, group_col, cat_col, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)

    g = tmpc[group_col].to_numpy()
    c = tmpc[cat_col].to_numpy()

    def chi2_stat(gvec):
        ct = pd.crosstab(c, gvec)
        return chi2_contingency(ct, correction=False)[0]

    obs = chi2_stat(g)
    perm_stats = np.array([chi2_stat(rng.permutation(g)) for _ in range(n_perm)])
    p = (perm_stats >= obs).mean()
    return obs, perm_stats, p

chi2_obs, perm_stats_cat, pval_cat = perm_test_chi2(
    tmpc, "miss_demand", cat, n_perm=5000, seed=42
)

chi2_obs, pval_cat

(np.float64(101.3661710576918), np.float64(0.0))

The missingness of DEMAND.LOSS.MW depends on CAUSE.CATEGORY.<br>

In [14]:
import plotly.express as px

plot_df = df.loc[df["CAUSE.CATEGORY"].notna(), ["CAUSE.CATEGORY", "miss_demand"]].copy()

plot_df["group"] = plot_df["miss_demand"].map({
    True: "DEMAND.LOSS.MW missing",
    False: "DEMAND.LOSS.MW present"
})

fig = px.histogram(
    plot_df,
    x="CAUSE.CATEGORY",
    color="group",
    barmode="overlay",          # same style as your duration plot
    histnorm="probability",     # normalize to compare proportions
    title="CAUSE.CATEGORY by DEMAND.LOSS.MW Missingness"
)

fig.update_layout(
    xaxis_title="CAUSE.CATEGORY",
    yaxis_title="Proportion within category"
)

fig.show()

In [15]:
df.head()

,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,...,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND,miss_demand,OUTAGE.DURATION_NUM
1,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,"Friday, July 1, 2011",...,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983,True,3060.0
2,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,"Sunday, May 11, 2014",...,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983,True,1.0
3,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,"Tuesday, October 26, 2010",...,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983,True,3000.0
4,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,"Tuesday, June 19, 2012",...,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983,True,2550.0
5,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,"Saturday, July 18, 2015",...,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983,False,1740.0


In [16]:
df.columns

Index(['OBS', 'YEAR', 'MONTH', 'U.S._STATE', 'POSTAL.CODE', 'NERC.REGION',
       'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CLIMATE.CATEGORY',
       'OUTAGE.START.DATE', 'OUTAGE.START.TIME', 'OUTAGE.RESTORATION.DATE',
       'OUTAGE.RESTORATION.TIME', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL',
       'HURRICANE.NAMES', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW',
       'CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE',
       'TOTAL.PRICE', 'RES.SALES', 'COM.SALES', 'IND.SALES', 'TOTAL.SALES',
       'RES.PERCEN', 'COM.PERCEN', 'IND.PERCEN', 'RES.CUSTOMERS',
       'COM.CUSTOMERS', 'IND.CUSTOMERS', 'TOTAL.CUSTOMERS', 'RES.CUST.PCT',
       'COM.CUST.PCT', 'IND.CUST.PCT', 'PC.REALGSP.STATE', 'PC.REALGSP.USA',
       'PC.REALGSP.REL', 'PC.REALGSP.CHANGE', 'UTIL.REALGSP', 'TOTAL.REALGSP',
       'UTIL.CONTRI', 'PI.UTIL.OFUSA', 'POPULATION', 'POPPCT_URBAN',
       'POPPCT_UC', 'POPDEN_URBAN', 'POPDEN_UC', 'POPDEN_RURAL',
       'AREAPCT_URBAN', 'AREAPCT_UC', 'PCT_LAND', 'PCT_WATER_TOT',
    

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1534 entries, 1 to 1534
Data columns (total 58 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   OBS                      1534 non-null   float64
 1   YEAR                     1534 non-null   float64
 2   MONTH                    1525 non-null   float64
 3   U.S._STATE               1534 non-null   object 
 4   POSTAL.CODE              1534 non-null   object 
 5   NERC.REGION              1534 non-null   object 
 6   CLIMATE.REGION           1528 non-null   object 
 7   ANOMALY.LEVEL            1525 non-null   object 
 8   CLIMATE.CATEGORY         1525 non-null   object 
 9   OUTAGE.START.DATE        1525 non-null   object 
 10  OUTAGE.START.TIME        1525 non-null   object 
 11  OUTAGE.RESTORATION.DATE  1476 non-null   object 
 12  OUTAGE.RESTORATION.TIME  1476 non-null   object 
 13  CAUSE.CATEGORY           1534 non-null   object 
 14  CAUSE.CATEGORY.DETAIL   

In [ ]:
df = pd.read_csv('outage.csv', skiprows=5)
df = df.drop('variables', axis=1)
df = df.iloc[1:]

In [36]:
# ============================================
# STEP 0: SETUP
# ============================================
import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================
# STEP 1: LOAD DATA
# ============================================
print("="*70)
print("STEP 1: LOADING DATA")
print("="*70)

# ---- Change this to your file path ----
df = pd.read_csv('outage.csv', skiprows=5)
df = df.drop('variables', axis=1)
df = df.iloc[1:]

print(f"Loaded df: {df.shape}")
print(df.dtypes.head(20))

# Target
TARGET_COL = "DEMAND.LOSS.MW"

# ============================================
# STEP 2: BASIC CLEANING + TYPE FIXES
# ============================================
print("\n" + "="*70)
print("STEP 2: CLEANING + TYPE FIXES")
print("="*70)

# Helper: numeric coercion for columns stored as object
def to_num(series):
    # remove commas and stray whitespace
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False).str.strip(),
                         errors="coerce")

# Columns that should be numeric but often appear as object in this dataset
numeric_object_cols = [
    "OUTAGE.DURATION", "DEMAND.LOSS.MW",
    "RES.PRICE", "COM.PRICE", "IND.PRICE", "TOTAL.PRICE",
    "RES.SALES", "COM.SALES", "IND.SALES", "TOTAL.SALES",
    "RES.PERCEN", "COM.PERCEN", "IND.PERCEN",
    "RES.CUST.PCT", "COM.CUST.PCT", "IND.CUST.PCT",
    "PC.REALGSP.STATE", "PC.REALGSP.USA", "PC.REALGSP.REL", "PC.REALGSP.CHANGE",
    "UTIL.REALGSP", "TOTAL.REALGSP", "UTIL.CONTRI", "PI.UTIL.OFUSA",
    "PCT_LAND", "PCT_WATER_TOT", "PCT_WATER_INLAND",
    "CUSTOMERS.AFFECTED"
]
for c in numeric_object_cols:
    if c in df.columns:
        df[c] = to_num(df[c])

# Year/month should be integers
if "YEAR" in df.columns:
    df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").astype("Int64")
if "MONTH" in df.columns:
    df["MONTH"] = pd.to_numeric(df["MONTH"], errors="coerce").astype("Int64")

# Parse outage start time to extract hour-of-day (optional but useful)
def parse_hour(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    # common formats: "13:45", "1:45 PM", etc.
    try:
        dt = pd.to_datetime(s, errors="coerce")
        if pd.isna(dt):
            return np.nan
        return float(dt.hour)
    except Exception:
        return np.nan

if "OUTAGE.START.TIME" in df.columns:
    df["START_HOUR"] = df["OUTAGE.START.TIME"].apply(parse_hour)

# Parse start date (optional) -> day of year
if "OUTAGE.START.DATE" in df.columns:
    dt = pd.to_datetime(df["OUTAGE.START.DATE"], errors="coerce")
    df["START_DAYOFYEAR"] = dt.dt.dayofyear.astype("float32")

# ---- Target handling ----
# Keep rows with non-missing target only
df[TARGET_COL] = to_num(df[TARGET_COL]) if df[TARGET_COL].dtype == "object" else df[TARGET_COL]
df_model = df.loc[df[TARGET_COL].notna()].copy()
print(f"Rows with non-missing target: {len(df_model):,} / {len(df):,}")

# Log-transform target to reduce heavy tails
df_model["y"] = np.log1p(df_model[TARGET_COL].clip(lower=0))

# ============================================
# STEP 3: FEATURE SELECTION
# ============================================
print("\n" + "="*70)
print("STEP 3: FEATURE SELECTION")
print("="*70)

# IMPORTANT: If you're doing *pre-event* prediction, you should exclude CAUSE.* and HURRICANE.NAMES
# Here we INCLUDE them by default since your prompt didn't exclude them.
drop_cols = [
    "OBS",
    TARGET_COL,              # raw target
    "OUTAGE.RESTORATION.DATE","OUTAGE.RESTORATION.TIME"  # can leak info about duration
]

# If you believe OUTAGE.DURATION is outcome-adjacent / leakage for demand loss, drop it.
# It's often correlated and could "cheat" depending on your framing.
LEAKY = True
if LEAKY and "OUTAGE.DURATION" in df_model.columns:
    drop_cols.append("OUTAGE.DURATION")

X = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns])
y = df_model["y"].astype("float32").values

# Identify categorical vs numeric columns
cat_cols = [
    c for c in X.columns
    if X[c].dtype == "object" or str(X[c].dtype).startswith("string")
]
num_cols = [
    c for c in X.columns
    if c not in cat_cols
]

# Some known categoricals even if not object (safety)
for c in ["YEAR","MONTH"]:
    if c in X.columns and c not in num_cols:
        pass

print(f"Num cols: {len(num_cols)}")
print(f"Cat cols: {len(cat_cols)}")
print("Example num cols:", num_cols[:10])
print("Example cat cols:", cat_cols[:10])

# ============================================
# STEP 4: TRAIN/VAL/TEST SPLIT (time-aware-ish)
# ============================================
print("\n" + "="*70)
print("STEP 4: TRAIN/VAL/TEST SPLIT")
print("="*70)

# Prefer a time split: train on earlier years, test on later years (less leakage).
# If YEAR is missing/too sparse, fall back to random split.
if "YEAR" in df_model.columns and df_model["YEAR"].notna().mean() > 0.95:
    years = df_model["YEAR"].astype(int)
    cutoff = np.quantile(years.unique(), 0.75)  # rough: last ~25% years as test
    cutoff = int(cutoff)

    train_mask = years <= cutoff
    test_mask  = years > cutoff

    X_train_full = X.loc[train_mask].copy()
    y_train_full = y[train_mask.values]
    X_test       = X.loc[test_mask].copy()
    y_test       = y[test_mask.values]

    # Validation split from training portion
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=SEED
    )
    print(f"Time-ish split by YEAR cutoff={cutoff}:")
else:
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=0.15, random_state=SEED
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=SEED
    )
    print("Random split used (YEAR not suitable).")

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

# ============================================
# STEP 5: PREPROCESSING (One-hot + scaling)
# ============================================
print("\n" + "="*70)
print("STEP 5: PREPROCESSING")
print("="*70)

numeric_transformer = Pipeline(steps=[
    ("imputer", tf.keras.layers.Lambda(lambda z: z))  # placeholder (we impute via pandas below)
])

# We'll impute with pandas before encoding/scaling (simple, reliable).
# Numeric: median; Categorical: "MISSING"
for c in num_cols:
    med = X_train[c].median() if c in X_train.columns else np.nan
    X_train[c] = X_train[c].fillna(med)
    X_val[c]   = X_val[c].fillna(med)
    X_test[c]  = X_test[c].fillna(med)

for c in cat_cols:
    X_train[c] = X_train[c].fillna("MISSING").astype(str)
    X_val[c]   = X_val[c].fillna("MISSING").astype(str)
    X_test[c]  = X_test[c].fillna("MISSING").astype(str)

# ColumnTransformer for sklearn encoding/scaling
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=True, with_std=True), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder="drop"
)

# Fit on train only
X_train_proc = preprocess.fit_transform(X_train)
X_val_proc   = preprocess.transform(X_val)
X_test_proc  = preprocess.transform(X_test)

print(f"Processed shapes: train={X_train_proc.shape}, val={X_val_proc.shape}, test={X_test_proc.shape}")

# Save feature dimension for Keras input
INPUT_DIM = X_train_proc.shape[1]

# ============================================
# STEP 6: BUILD KERAS REGRESSION MODEL
# ============================================
print("\n" + "="*70)
print("STEP 6: BUILDING MODEL (tensorflow.keras)")
print("="*70)

L2 = 1e-5

inputs = layers.Input(shape=(INPUT_DIM,), name="features", dtype=tf.float32)

x = layers.Dense(256, activation="relu",
                 kernel_regularizer=regularizers.l2(L2))(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)

x = layers.Dense(128, activation="relu",
                 kernel_regularizer=regularizers.l2(L2))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.15)(x)

# Output is log1p(demand loss) regression
outputs = layers.Dense(1, activation="linear", name="y_hat")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=[tf.keras.metrics.MAE, tf.keras.metrics.MSE]
)

model.summary()
print(f"Total parameters: {model.count_params():,}")

# ============================================
# STEP 7: TF.DATA PIPELINES
# ============================================
print("\n" + "="*70)
print("STEP 7: TF.DATA PIPELINES")
print("="*70)

BATCH = 64

def make_ds(Xn, yn, training=False):
    ds = tf.data.Dataset.from_tensor_slices((Xn.astype("float32"), yn.astype("float32")))
    if training:
        ds = ds.shuffle(min(len(Xn), 10_000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(X_train_proc, y_train, training=True)
val_ds   = make_ds(X_val_proc, y_val, training=False)
test_ds  = make_ds(X_test_proc, y_test, training=False)

# ============================================
# STEP 8: TRAINING
# ============================================
print("\n" + "="*70)
print("STEP 8: TRAINING")
print("="*70)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/best_demand_loss_model.keras",
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,
    callbacks=callbacks,
    verbose=1
)

# ============================================
# STEP 9: EVALUATION
# ============================================
print("\n" + "="*70)
print("STEP 9: EVALUATION")
print("="*70)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load best model
best_model = tf.keras.models.load_model("/content/best_demand_loss_model.keras")

# Predictions
yhat_log = best_model.predict(test_ds).reshape(-1)
ytrue_log = y_test.reshape(-1)

# --------------------------------
# LOG SPACE METRICS
# --------------------------------
mae_log = mean_absolute_error(ytrue_log, yhat_log)
rmse_log = np.sqrt(mean_squared_error(ytrue_log, yhat_log))
r2_log = r2_score(ytrue_log, yhat_log)

print("\nLOG SPACE METRICS")
print("-"*50)
print(f"MAE  (log): {mae_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"R²   (log): {r2_log:.4f}")

# --------------------------------
# ORIGINAL MW SPACE METRICS
# --------------------------------
yhat_mw = np.expm1(yhat_log)
ytrue_mw = np.expm1(ytrue_log)

mae_mw = mean_absolute_error(ytrue_mw, yhat_mw)
rmse_mw = np.sqrt(mean_squared_error(ytrue_mw, yhat_mw))
r2_mw = r2_score(ytrue_mw, yhat_mw)

print("\nORIGINAL MW SPACE METRICS")
print("-"*50)
print(f"MAE  (MW): {mae_mw:,.3f}")
print(f"RMSE (MW): {rmse_mw:,.3f}")
print(f"R²   (MW): {r2_mw:.4f}")

# --------------------------------
# EXTRA DIAGNOSTICS
# --------------------------------
print("\nPREDICTION DIAGNOSTICS")
print("-"*50)
print(f"Prediction range: [{yhat_mw.min():.2f}, {yhat_mw.max():.2f}]")
print(f"True range:       [{ytrue_mw.min():.2f}, {ytrue_mw.max():.2f}]")
print(f"Prediction mean:  {yhat_mw.mean():.2f}")
print(f"Actual mean:      {ytrue_mw.mean():.2f}")
# ============================================
# STEP 10: (OPTIONAL) VARIABLE-LEVEL IMPORTANCE TEST (Permutation Importance)
# ============================================
print("\n" + "="*70)
print("STEP 10: OPTIONAL PERMUTATION IMPORTANCE (BLOCKED)")
print("="*70)

# We'll do "single-variable blocked permutation" on the TEST set:
# permute one column within (U.S._STATE, MONTH) blocks, re-transform, re-score MAE.
# This estimates how much that variable matters for prediction.

def score_mae_on_df(df_X, y_log_true, batch_size=256):
    """
    Impute like training, transform via preprocess, predict with best_model,
    return MAE in log space.
    """
    tmp = df_X.copy()

    # Impute numeric using TRAIN medians
    for c in num_cols:
        med = X_train[c].median()
        tmp[c] = tmp[c].fillna(med)

    # Impute categoricals
    for c in cat_cols:
        tmp[c] = tmp[c].fillna("MISSING").astype(str)

    Xp = preprocess.transform(tmp).astype("float32")
    ds = tf.data.Dataset.from_tensor_slices(Xp).batch(batch_size)
    yhat = best_model.predict(ds, verbose=0).reshape(-1)

    return float(np.mean(np.abs(yhat - y_log_true)))

baseline_mae = score_mae_on_df(X_test, ytrue_log)
print(f"Baseline test MAE (log): {baseline_mae:.4f}")

def blocked_permute_column(df_X, col, block_cols=("U.S._STATE", "MONTH"), seed=None):
    """
    Permute column 'col' *within* blocks defined by block_cols.
    Uses positional indexing (iloc) to avoid index-label KeyErrors.
    """
    rng = np.random.default_rng(seed)
    tmp = df_X.copy()

    # Fallback if block columns missing
    if not all(b in tmp.columns for b in block_cols):
        tmp[col] = rng.permutation(tmp[col].to_numpy())
        return tmp

    # Group by blocks; indices returned are *positional* indices into the original object
    gb = tmp.groupby(list(block_cols), dropna=False, sort=False)

    for _, idx in gb.indices.items():
        idx = np.asarray(idx)  # positional integer indices
        vals = tmp.iloc[idx][col].to_numpy()
        tmp.iloc[idx, tmp.columns.get_loc(col)] = rng.permutation(vals)

    return tmp

# Example: run importance for a few candidate variables
CANDIDATES = [
    "CLIMATE.CATEGORY",
    "ANOMALY.LEVEL",
    "TOTAL.SALES",
    "TOTAL.CUSTOMERS",
    "POPDEN_URBAN",
    "PC.REALGSP.STATE",
    "CAUSE.CATEGORY"
]
CANDIDATES = [c for c in CANDIDATES if c in X_test.columns]

B = 50  # increase to 200+ for nicer p-values
for col in CANDIDATES:
    maes = []
    for _ in range(B):
        Xp = blocked_permute_column(X_test, col, block_cols=("U.S._STATE","MONTH"))
        maes.append(score_mae_on_df(Xp, ytrue_log))
    maes = np.array(maes)
    delta = maes - baseline_mae
    print(f"\n{col}")
    print(f"  mean ΔMAE: {delta.mean():.4f}")
    print(f"  95% ΔMAE: [{np.quantile(delta,0.025):.4f}, {np.quantile(delta,0.975):.4f}]")
    # one-sided p-value: how often permuting does NOT worsen error (Δ<=0)
    p = (1 + np.sum(delta <= 0)) / (1 + B)
    print(f"  p-value (importance): {p:.4f}")

print("\nDone.")

STEP 1: LOADING DATA
Loaded df: (1534, 56)
OBS                        float64
YEAR                       float64
MONTH                      float64
U.S._STATE                  object
POSTAL.CODE                 object
NERC.REGION                 object
CLIMATE.REGION              object
ANOMALY.LEVEL               object
CLIMATE.CATEGORY            object
OUTAGE.START.DATE           object
OUTAGE.START.TIME           object
OUTAGE.RESTORATION.DATE     object
OUTAGE.RESTORATION.TIME     object
CAUSE.CATEGORY              object
CAUSE.CATEGORY.DETAIL       object
HURRICANE.NAMES             object
OUTAGE.DURATION             object
DEMAND.LOSS.MW              object
CUSTOMERS.AFFECTED         float64
RES.PRICE                   object
dtype: object

STEP 2: CLEANING + TYPE FIXES
Rows with non-missing target: 829 / 1,534

STEP 3: FEATURE SELECTION
Num cols: 36
Cat cols: 18
Example num cols: ['YEAR', 'MONTH', 'CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE', 'TOTAL.PRICE', 'RES.

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 1275)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 256)            │       326,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_38 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ y_hat (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 369,409 (1.41 MB)

 Trainable params: 368,641 (1.41 MB)

 Non-trainable params: 768 (3.00 KB)

Total parameters: 369,409

STEP 7: TF.DATA PIPELINES

STEP 8: TRAINING
Epoch 1/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 5s 92ms/step - loss: 3.5765 - mean_absolute_error: 4.0469 - mean_squared_error: 21.7601 - val_loss: 3.4231 - val_mean_absolute_error: 3.8528 - val_mean_squared_error: 19.1040 - learning_rate: 0.0010
Epoch 2/200
1/9 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 2.8307 - mean_absolute_error: 3.3028 - mean_squared_error: 14.1626

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning:

Early stopping conditioned on metric `val_mae` which is not available. Available metrics are: loss,mean_absolute_error,mean_squared_error,val_loss,val_mean_absolute_error,val_mean_squared_error

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/callback_list.py:145: UserWarning:

Learning rate reduction is conditioned on metric `val_mae` which is not available. Available metrics are: loss,mean_absolute_error,mean_squared_error,val_loss,val_mean_absolute_error,val_mean_squared_error,learning_rate.

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/model_checkpoint.py:302: UserWarning:

Can save best model only with val_mae available.



9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 2.1891 - mean_absolute_error: 2.6446 - mean_squared_error: 10.2335 - val_loss: 3.0838 - val_mean_absolute_error: 3.5167 - val_mean_squared_error: 15.9250 - learning_rate: 0.0010
Epoch 3/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.3186 - mean_absolute_error: 1.7570 - mean_squared_error: 4.9491 - val_loss: 2.7996 - val_mean_absolute_error: 3.2395 - val_mean_squared_error: 13.4527 - learning_rate: 0.0010
Epoch 4/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 1.0969 - mean_absolute_error: 1.5188 - mean_squared_error: 3.6007 - val_loss: 2.5896 - val_mean_absolute_error: 3.0400 - val_mean_squared_error: 11.7880 - learning_rate: 0.0010
Epoch 5/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.8609 - mean_absolute_error: 1.2743 - mean_squared_error: 2.6877 - val_loss: 2.4812 - val_mean_absolute_error: 2.9315 - val_mean_squared_error: 10.9068 - learning_rate: 0.0010
Epoch 6/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.7122 - mean

In [34]:
# ============================================
# XGBOOST REGRESSION + GRIDSEARCHCV
# Predict DEMAND.LOSS.MW using log1p target
# ============================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin, clone

# ---- XGBoost ----
from xgboost import XGBRegressor

SEED = 42
np.random.seed(SEED)

# ============================================
# STEP 1: LOAD DATA
# ============================================
print("="*70)
print("STEP 1: LOADING DATA")
print("="*70)

df = pd.read_csv('outage.csv', skiprows=5)
df = df.drop('variables', axis=1)
df = df.iloc[1:]
print(f"Loaded df: {df.shape}")

TARGET_COL = "DEMAND.LOSS.MW"

# ============================================
# STEP 2: CLEANING / TYPE FIXES
# ============================================
print("\n" + "="*70)
print("STEP 2: CLEANING / TYPE FIXES")
print("="*70)

def to_num(series):
    # handle commas, whitespace, and non-numeric junk
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

# Convert known numeric-in-object columns if they exist
numeric_object_cols = [
    "OUTAGE.DURATION", "DEMAND.LOSS.MW",
    "RES.PRICE", "COM.PRICE", "IND.PRICE", "TOTAL.PRICE",
    "RES.SALES", "COM.SALES", "IND.SALES", "TOTAL.SALES",
    "RES.PERCEN", "COM.PERCEN", "IND.PERCEN",
    "RES.CUST.PCT", "COM.CUST.PCT", "IND.CUST.PCT",
    "PC.REALGSP.STATE", "PC.REALGSP.USA", "PC.REALGSP.REL", "PC.REALGSP.CHANGE",
    "UTIL.REALGSP", "TOTAL.REALGSP", "UTIL.CONTRI", "PI.UTIL.OFUSA",
    "POPULATION", "POPPCT_URBAN", "POPPCT_UC", "POPDEN_URBAN", "POPDEN_UC", "POPDEN_RURAL",
    "AREAPCT_URBAN", "AREAPCT_UC", "PCT_LAND", "PCT_WATER_TOT", "PCT_WATER_INLAND",
    "CUSTOMERS.AFFECTED"
]
for c in numeric_object_cols:
    if c in df.columns and df[c].dtype == "object":
        df[c] = to_num(df[c])

# YEAR/MONTH to numeric
if "YEAR" in df.columns:
    df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")
if "MONTH" in df.columns:
    df["MONTH"] = pd.to_numeric(df["MONTH"], errors="coerce")

# Extract hour/day-of-year from start time/date (optional but usually helpful)
def parse_hour(x):
    if pd.isna(x):
        return np.nan
    dt = pd.to_datetime(str(x).strip(), errors="coerce")
    return np.nan if pd.isna(dt) else float(dt.hour)

if "OUTAGE.START.TIME" in df.columns:
    df["START_HOUR"] = df["OUTAGE.START.TIME"].apply(parse_hour)

if "OUTAGE.START.DATE" in df.columns:
    dt = pd.to_datetime(df["OUTAGE.START.DATE"], errors="coerce")
    df["START_DAYOFYEAR"] = dt.dt.dayofyear

# Target clean
if TARGET_COL in df.columns and df[TARGET_COL].dtype == "object":
    df[TARGET_COL] = to_num(df[TARGET_COL])

df = df.loc[df[TARGET_COL].notna()].copy()
df[TARGET_COL] = df[TARGET_COL].clip(lower=0).astype("float32")
print(f"Rows with non-missing target: {len(df):,}")

# ============================================
# STEP 3: FEATURE SET (leakage-aware)
# ============================================
print("\n" + "="*70)
print("STEP 3: FEATURE SET")
print("="*70)

# If you consider OUTAGE.DURATION as leakage/outcome-adjacent, drop it.
LEAKY = True

drop_cols = [
    "OBS",
    TARGET_COL,
    "OUTAGE.RESTORATION.DATE", "OUTAGE.RESTORATION.TIME",
    "miss_demand"
]
if LEAKY and "OUTAGE.DURATION" in df.columns:
    drop_cols.append("OUTAGE.DURATION")

X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
y_mw = df[TARGET_COL].values.astype("float32")
y_log = np.log1p(y_mw).astype("float32")

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

print(f"Num cols: {len(num_cols)}")
print(f"Cat cols: {len(cat_cols)}")

# ============================================
# STEP 4: TRAIN/VAL/TEST SPLIT (time-aware if possible)
# ============================================
print("\n" + "="*70)
print("STEP 4: TRAIN/VAL/TEST SPLIT")
print("="*70)

def time_split_by_year(df_full, test_frac=0.20, val_frac=0.20):
    if "YEAR" not in df_full.columns or df_full["YEAR"].notna().mean() < 0.95:
        idx = np.arange(len(df_full))
        tr, te = train_test_split(idx, test_size=test_frac, random_state=SEED)
        tr, va = train_test_split(tr, test_size=val_frac, random_state=SEED)
        return tr, va, te, None

    years = df_full["YEAR"].astype(float).to_numpy()
    uniq = np.sort(pd.Series(years).dropna().unique())
    if len(uniq) < 5:
        idx = np.arange(len(df_full))
        tr, te = train_test_split(idx, test_size=test_frac, random_state=SEED)
        tr, va = train_test_split(tr, test_size=val_frac, random_state=SEED)
        return tr, va, te, None

    cutoff_idx = int(np.floor((1.0 - test_frac) * len(uniq))) - 1
    cutoff_year = int(uniq[max(cutoff_idx, 0)])

    te_mask = years > cutoff_year
    tr_mask = ~te_mask

    tr_idx = np.where(tr_mask)[0]
    te_idx = np.where(te_mask)[0]
    tr_idx, va_idx = train_test_split(tr_idx, test_size=val_frac, random_state=SEED)
    return tr_idx, va_idx, te_idx, cutoff_year

train_idx, val_idx, test_idx, cutoff_year = time_split_by_year(df, test_frac=0.20, val_frac=0.20)
print(f"Train: {len(train_idx):,}  Val: {len(val_idx):,}  Test: {len(test_idx):,}")
if cutoff_year is not None:
    print(f"Time cutoff YEAR: {cutoff_year} (test years > cutoff)")

X_train, X_val, X_test = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
y_train_log, y_val_log, y_test_log = y_log[train_idx], y_log[val_idx], y_log[test_idx]
y_test_mw = y_mw[test_idx]

# We'll do hyperparam search on TRAIN+VAL (to use CV), keep TEST untouched.
X_tune = pd.concat([X_train, X_val], axis=0)
y_tune = np.concatenate([y_train_log, y_val_log], axis=0)

# ============================================
# STEP 5: PREPROCESSING PIPELINE
# ============================================
print("\n" + "="*70)
print("STEP 5: PREPROCESSING PIPELINE")
print("="*70)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

# ============================================
# STEP 6: MODEL + GRID SEARCH
# ============================================
print("\n" + "="*70)
print("STEP 6: GRID SEARCH (XGBoost)")
print("="*70)

xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=SEED,
    tree_method="hist",     # fast on CPU
    n_jobs=-1,
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", xgb)
])

# Compact, practical grid (won't take forever)
param_grid = {
    "model__n_estimators": [400, 800],
    "model__learning_rate": [0.03, 0.07, 0.10],
    "model__max_depth": [3, 5, 7],
    "model__min_child_weight": [1, 5, 10],
    "model__subsample": [0.7, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.9, 1.0],
    "model__reg_alpha": [0.0, 1e-3, 1e-2],
    "model__reg_lambda": [1.0, 3.0, 10.0],
}

# Choose CV: if YEAR is reliable, do TimeSeriesSplit over the TUNE set sorted by YEAR
if cutoff_year is not None and "YEAR" in X_tune.columns:
    tune_order = np.argsort(df.iloc[np.concatenate([train_idx, val_idx])]["YEAR"].astype(float).values)
    X_tune = X_tune.iloc[tune_order]
    y_tune = y_tune[tune_order]

    cv = TimeSeriesSplit(n_splits=5)
    print("Using TimeSeriesSplit(5) on YEAR-sorted tuning set.")
else:
    cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
    print("Using shuffled KFold(5).")

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",   # MAE in LOG space
    cv=cv,
    verbose=2,
    n_jobs=-1,
)

grid.fit(X_tune, y_tune)

print("\nBest params:")
print(grid.best_params_)
print(f"Best CV score (neg MAE log): {grid.best_score_:.4f}")

best_model = grid.best_estimator_

# ============================================
# STEP 7: EVALUATION ON HELD-OUT TEST
# ============================================
print("\n" + "="*70)
print("STEP 7: EVALUATION (Held-out TEST)")
print("="*70)

yhat_test_log = best_model.predict(X_test).astype("float32")
yhat_test_mw = np.maximum(0.0, np.expm1(yhat_test_log)).astype("float32")

# MW space metrics
mae_mw = mean_absolute_error(y_test_mw, yhat_test_mw)
rmse_mw = np.sqrt(mean_squared_error(y_test_mw, yhat_test_mw))
r2_mw = r2_score(y_test_mw, yhat_test_mw)

print("\nMW SPACE METRICS")
print("-"*50)
print(f"MAE  (MW): {mae_mw:,.3f}")
print(f"RMSE (MW): {rmse_mw:,.3f}")
print(f"R²   (MW): {r2_mw:.4f}")

# log space metrics
mae_log = mean_absolute_error(y_test_log, yhat_test_log)
rmse_log = np.sqrt(mean_squared_error(y_test_log, yhat_test_log))
r2_log = r2_score(y_test_log, yhat_test_log)

print("\nLOG1P(MW) SPACE METRICS")
print("-"*50)
print(f"MAE  (log): {mae_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"R²   (log): {r2_log:.4f}")

print("\nDIAGNOSTICS")
print("-"*50)
print(f"Pred MW range: [{yhat_test_mw.min():.2f}, {yhat_test_mw.max():.2f}]")
print(f"True MW range: [{y_test_mw.min():.2f}, {y_test_mw.max():.2f}]")
print(f"Pred MW mean:  {yhat_test_mw.mean():.2f}")
print(f"True MW mean:  {y_test_mw.mean():.2f}")

# Tail (top 10% by true MW)
q90_true = np.quantile(y_test_mw, 0.90)
tail_mask = y_test_mw >= q90_true
if tail_mask.sum() > 0:
    tail_mae = mean_absolute_error(y_test_mw[tail_mask], yhat_test_mw[tail_mask])
    tail_rmse = np.sqrt(mean_squared_error(y_test_mw[tail_mask], yhat_test_mw[tail_mask]))
    print("\nTAIL (Top 10% True MW)")
    print("-"*50)
    print(f"Count: {tail_mask.sum():,}  Threshold: {q90_true:.1f} MW")
    print(f"Tail MAE:  {tail_mae:,.3f}")
    print(f"Tail RMSE: {tail_rmse:,.3f}")

# ============================================
# STEP 8: (Optional) FEATURE IMPORTANCE
# (XGBoost has gain-based importances; beware one-hot expansion)
# ============================================
print("\n" + "="*70)
print("STEP 8: OPTIONAL FEATURE IMPORTANCE (Top 30)")
print("="*70)

# Get expanded feature names from preprocessing
prep = best_model.named_steps["prep"]
feature_names = []
if len(num_cols) > 0:
    feature_names.extend(num_cols)

if len(cat_cols) > 0:
    ohe = prep.named_transformers_["cat"].named_steps["onehot"]
    ohe_names = ohe.get_feature_names_out(cat_cols).tolist()
    feature_names.extend(ohe_names)

booster = best_model.named_steps["model"]
importances = booster.feature_importances_

# Safety check
k = min(30, len(importances))
top_idx = np.argsort(importances)[::-1][:k]
for i in top_idx:
    print(f"{feature_names[i]:<45}  importance={importances[i]:.6f}")

print("\nDone.")

STEP 1: LOADING DATA
Loaded df: (1534, 56)

STEP 2: CLEANING / TYPE FIXES
Rows with non-missing target: 829

STEP 3: FEATURE SET
Num cols: 42
Cat cols: 11

STEP 4: TRAIN/VAL/TEST SPLIT
Train: 530  Val: 133  Test: 166
Time cutoff YEAR: 2012 (test years > cutoff)

STEP 5: PREPROCESSING PIPELINE

STEP 6: GRID SEARCH (XGBoost)
Using TimeSeriesSplit(5) on YEAR-sorted tuning set.
Fitting 5 folds for each of 4374 candidates, totalling 21870 fits

Best params:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.03, 'model__max_depth': 7, 'model__min_child_weight': 1, 'model__n_estimators': 400, 'model__reg_alpha': 0.001, 'model__reg_lambda': 1.0, 'model__subsample': 0.7}
Best CV score (neg MAE log): -0.6397

STEP 7: EVALUATION (Held-out TEST)

MW SPACE METRICS
--------------------------------------------------
MAE  (MW): 602.063
RMSE (MW): 3,612.508
R²   (MW): -0.0012

LOG1P(MW) SPACE METRICS
--------------------------------------------------
MAE  (log): 0.6740
RMSE (log): 1.1949
R²   (